# Step 3: Data Preprocessing

This notebook handles **Data Preprocessing**:

1. **Preprocessing**: Computes engineered features from raw data
2. **Train/Test Split**: Creates separate tables for model training

## Prerequisites

- Run notebooks 01-02 first

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

# try:
#     session.sql("USE ROLE ACCOUNTADMIN").collect()
# except:
#     print(f"Using current role: {session.get_current_role()}")


DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## Read Source Data

In [ ]:
print(f"Reading data from {config.full_raw_table}...")
source_df = session.table(config.full_raw_table)
source_df.show(5)

## Create Feature Store

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

fs = FeatureStore(
    session=session, 
    database=DB, 
    name=SCHEMA, 
    default_warehouse=COMPUTE_WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

fs.list_entities().to_pandas()

## Feature Engineering

In [ ]:
import snowflake.snowpark.functions as F

feature_df = (
    source_df
    .with_column("SHOCK_INDEX", F.col("HEART_RATE") / F.col("SYSTOLIC_BP"))
    .with_column("PULSE_PRESSURE", F.col("SYSTOLIC_BP") - F.col("DIASTOLIC_BP"))
    .with_column(
        "BMI_CATEGORY",
        F.when(F.col("BMI") < 18.5, F.lit("UNDERWEIGHT"))
         .when(F.col("BMI") < 25.0, F.lit("NORMAL"))
         .when(F.col("BMI") < 30.0, F.lit("OVERWEIGHT"))
         .otherwise(F.lit("OBESE")),
    )
    .with_column(
        "VITAL_SIGNS_SEVERITY",
        (
            F.when((F.col("HEART_RATE") > 100) | (F.col("HEART_RATE") < 50), F.lit(1)).otherwise(F.lit(0))
            + F.when((F.col("SYSTOLIC_BP") > 180) | (F.col("SYSTOLIC_BP") < 90), F.lit(2)).otherwise(F.lit(0))
            + F.when(F.col("OXYGEN_SATURATION") < 92, F.lit(2)).otherwise(
                F.when(F.col("OXYGEN_SATURATION") < 95, F.lit(1)).otherwise(F.lit(0))
            )
            + F.when(F.col("RESPIRATORY_RATE") > 24, F.lit(1)).otherwise(F.lit(0))
            + F.when((F.col("TEMPERATURE") > 38.5) | (F.col("TEMPERATURE") < 36.0), F.lit(1)).otherwise(F.lit(0))
        ),
    )
)

feature_df.select(
    "PATIENT_ID", "TIMESTAMP",
    "SHOCK_INDEX", "PULSE_PRESSURE", "BMI_CATEGORY", "VITAL_SIGNS_SEVERITY",
).show(5)

## Create Feature Store Entity

In [ ]:
patient_entity = Entity(
    name="PATIENT",
    join_keys=["PATIENT_ID"],
    desc="Hospital patient identified by PATIENT_ID",
)
fs.register_entity(patient_entity)

## Create and Populate Feature View

In [ ]:
fv = FeatureView(
    name="PATIENT_FEATURES",
    entities=[patient_entity],
    feature_df=feature_df,
    timestamp_col="TIMESTAMP",
    refresh_freq="1 minute",
    desc=(
        "Raw vitals + 4 engineered features: "
        "SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY"
    ),
)

fv.attach_feature_desc(
    {
        "SHOCK_INDEX": "Ratio of heart rate to systolic blood pressure, indicating hemodynamic instability",
        "PULSE_PRESSURE": "Difference between systolic and diastolic blood pressure",
        "BMI_CATEGORY": "Categorical classification of BMI: UNDERWEIGHT, NORMAL, OVERWEIGHT, OBESE",
        "VITAL_SIGNS_SEVERITY": "Composite severity score based on abnormal vital sign thresholds",
    }
)

registered_fv = fs.register_feature_view(
    feature_view=fv,
    version="v1",
    block=True,
    overwrite=True,
)

print("Entities in Feature Store:")
entities_df = fs.list_entities().to_pandas()
display(entities_df if len(entities_df) > 0 else "No entities found")

print("\nFeature Views in Feature Store:")
fv_df = fs.list_feature_views().to_pandas()
display(fv_df[["NAME", "VERSION", "DESC", "REFRESH_FREQ", "SCHEDULING_STATE"]] if len(fv_df) > 0 else "No feature views found")

## Snowpark-Native Train/Test Split

In [ ]:
training_dataset_name = "PATIENT_RISK_TRAINING_DATASET"
training_dataset_version = datetime.now(timezone.utc).strftime("v_%Y%m%d_%H%M%S")
test_table = f"{DB}.{SCHEMA}.TEST_FEATURES"

spine_df = (
    session.table(config.full_raw_table)
    .select("PATIENT_ID", "TIMESTAMP")
    .with_column("_SPLIT", F.uniform(F.lit(0), F.lit(1), F.random()))
)
train_spine = spine_df.filter(F.col("_SPLIT") < 0.8).drop("_SPLIT")
test_spine = spine_df.filter(F.col("_SPLIT") >= 0.8).drop("_SPLIT")

training_dataset = fs.generate_dataset(
    spine_df=train_spine,
    features=[registered_fv],
    spine_timestamp_col="TIMESTAMP",
    name=training_dataset_name,
    version=training_dataset_version,
    desc=f"Point-in-time training snapshot for PATIENT_RISK model — {training_dataset_version}",
)

test_df = fs.retrieve_feature_values(
    spine_df=test_spine,
    features=[registered_fv],
    spine_timestamp_col="TIMESTAMP",
)
test_df.write.mode("overwrite").save_as_table(test_table)

training_rows = training_dataset.read.to_snowpark_dataframe().count()
test_rows = session.table(test_table).count()
print(f"Training dataset: {training_dataset_name}/{training_dataset_version}  ({training_rows:,} rows)")
print(f"Test table:       {test_table}  ({test_rows:,} rows)")

## Verify Outputs

In [ ]:
print("Sample from training dataset:")
display(
    training_dataset.read.to_snowpark_dataframe().select(
        "PATIENT_ID", "HEART_RATE", "SYSTOLIC_BP", "DIASTOLIC_BP",
        "SHOCK_INDEX", "PULSE_PRESSURE", "BMI_CATEGORY", "VITAL_SIGNS_SEVERITY", "RISK_LEVEL",
    ).limit(5).to_pandas()
)

In [ ]:
print("Sample from TEST_FEATURES table:")
display(session.sql(f"""
    SELECT PATIENT_ID, HEART_RATE, SYSTOLIC_BP, DIASTOLIC_BP,
           SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY, RISK_LEVEL
    FROM {test_table}
    LIMIT 5
""").to_pandas())